# OrientedDet × FAIR1M (Kaggle)

End-to-end tutorial for **fine-grained oriented detection** on the public FAIR1M train dump.

| | |
|---|---|
| Dataset | [`ollypowell/fair1m-satellite-imagery-for-object-detection`](https://www.kaggle.com/datasets/ollypowell/fair1m-satellite-imagery-for-object-detection) (~9 GB JPG) |
| License | **CC BY-NC-SA 3.0 IGO** (non-commercial). OrientedDet does **not** ship Hub weights for FAIR1M. |
| Library | [DL4EO/oriented-det](https://github.com/DL4EO/oriented-det) |

### Kaggle setup

1. **Add Input** → `fair1m-satellite-imagery-for-object-detection` (ollypowell)
2. **Accelerator** → GPU (T4/P100) if you enable the train smoke; CPU is enough for convert/tile
3. **Internet** → ON (for `pip install`)
4. Defaults use a **small image subset** so convert + tile fit in `/kaggle/working`. Set `SUBSET_IMAGES = None` only if you have enough disk/time.

**Pipeline:** discover → visualize → `fair1m-to-dota` (image-level holdout) → `tile-dota` (1024 / overlap 200) → short `odet train` smoke.

**Do not expect DOTA-like mAP.** FAIR1M is 37 fine-grained classes. A full 1× Rotated Faster R-CNN finetune from the DOTA Hub checkpoint reached **36.7%** mAP50 on tiled val (still climbing). Literature Faster R-CNN sits in the low-to-mid 30s. This notebook’s 1-epoch smoke will not hit that number — see **Expected mAP** below.


## 0. Install OrientedDet


In [ ]:
# After FAIR1M lands on main, leave REPO_REF empty.
# Until then, pin a commit/branch that includes FAIR1M, e.g. REPO_REF = "@main"
REPO_REF = ""

# Optional: if you uploaded this repo as a Kaggle Dataset, install editable from input instead:
#   SRC = next(Path("/kaggle/input").glob("*oriented*"), None)
#   if SRC: %pip install -q -e {SRC}

%pip install -q "git+https://github.com/DL4EO/oriented-det.git{0}".format(REPO_REF)

import oriented_det
from oriented_det.data import FAIR1M_CLASSES, FAIR1M_GROUPS

print("oriented_det", getattr(oriented_det, "__version__", "?"))
print(f"{len(FAIR1M_CLASSES)} fine classes, {len(FAIR1M_GROUPS)} coarse groups")
assert hasattr(oriented_det.data, "FAIR1MDataset"), (
    "Installed build has no FAIR1M loader — set REPO_REF to a FAIR1M-capable commit/branch."
)


## 1. Paths & knobs


In [ ]:
from __future__ import annotations

import json
import os
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from PIL import Image

# --- Knobs (safe for a typical Kaggle session) ---
SUBSET_IMAGES = 48       # None = full train dump (slow / disk-heavy)
VAL_FRACTION = 0.1
SPLIT_SEED = 0
TILE_SIZE = 1024
TILE_OVERLAP = 200       # pixels
MIN_BOX_OVERLAP = 0.7
RUN_TRAIN = True
MAX_TRAIN_SAMPLES = 64   # tile cap for smoke train
MAX_VAL_SAMPLES = 16
NUM_EPOCHS = 1
BATCH_SIZE = 2
NUM_WORKERS = 2

KAGGLE_INPUT = Path("/kaggle/input")
IS_KAGGLE = KAGGLE_INPUT.is_dir()

if IS_KAGGLE:
    WORK = Path("/kaggle/working")
    WORK.mkdir(parents=True, exist_ok=True)
    candidates = [
        KAGGLE_INPUT / "fair1m-satellite-imagery-for-object-detection",
        *sorted(KAGGLE_INPUT.glob("*fair1m*")),
        *sorted(p for p in KAGGLE_INPUT.iterdir() if p.is_dir()),
    ]
    DATA_ROOT = None
    for c in candidates:
        if (c / "Dataset" / "Images" / "Train").is_dir() or (c / "train" / "part1").is_dir():
            DATA_ROOT = c
            break
    if DATA_ROOT is None:
        raise FileNotFoundError(
            "FAIR1M not found under /kaggle/input. "
            "Add dataset ollypowell/fair1m-satellite-imagery-for-object-detection."
        )
else:
    DATA_ROOT = Path(os.environ.get("FAIR1M_ROOT", "./data/FAIR1M"))
    WORK = Path(os.environ.get("FAIR1M_WORK", "./fair1m_work"))
    WORK.mkdir(parents=True, exist_ok=True)

DOTA_DIR = WORK / "FAIR1M-dota"
CONFIG_PATH = WORK / "fair1m_kaggle_smoke.json"

# Runs land under WORK/runs (not site-packages)
os.environ["ORIENTED_DET_PROJECT_ROOT"] = str(WORK.resolve())
os.chdir(WORK)

print("DATA_ROOT =", DATA_ROOT)
print("WORK      =", WORK)
print("CUDA     =", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
print("free GB  ≈", round(shutil.disk_usage(WORK).free / 1e9, 1))


## 2. Discover layout & class counts


In [ ]:
from oriented_det.data.fair1m import (
    FAIR1MDataset,
    discover_fair1m_pairs,
    resolve_fair1m_root,
)

root = resolve_fair1m_root(DATA_ROOT)
pairs = discover_fair1m_pairs(DATA_ROOT, "train")
labeled = [(img, xml) for img, xml in pairs if xml is not None]
print(f"Resolved root: {root}")
print(f"Train images: {len(pairs)}  with XML: {len(labeled)}")
if not labeled:
    raise RuntimeError(
        "No image/XML pairs found. Expect Dataset/Images/Train/*.jpg with either "
        "Dataset/Labels/Train/*.xml (same stem) or Notebook_Working/train_labels/N.xml "
        "(JPG stem t_N → XML stem N)."
    )
pairs = labeled

scan_n = min(len(pairs), SUBSET_IMAGES or 200, 200)
ds_scan = FAIR1MDataset(
    DATA_ROOT,
    split="train",
    difficult_strategy="keep",
    stem_list=[p[0].stem for p in pairs[:scan_n]],
)
counts: Counter[str] = Counter()
for sample in ds_scan:
    for ann in sample.annotations:
        counts[ann.class_name] += 1

print(f"Scanned {scan_n} images → {sum(counts.values())} boxes, {len(counts)} classes")
for name, n in counts.most_common(15):
    print(f"  {n:6d}  {name}")
if len(counts) > 15:
    print(f"  ... ({len(counts) - 15} more)")


## 3. Visualize oriented boxes


In [ ]:
from oriented_det.geometry import RBox
from oriented_det.utils.viz import draw_boxes

viz_stems = [p[0].stem for p in pairs[:3]]
ds_viz = FAIR1MDataset(
    DATA_ROOT, split="train", difficult_strategy="keep", stem_list=viz_stems
)

fig, axes = plt.subplots(1, len(viz_stems), figsize=(5 * len(viz_stems), 5))
if len(viz_stems) == 1:
    axes = [axes]

for ax, sample in zip(axes, ds_viz):
    img = Image.open(sample.image_path).convert("RGB")
    max_side = 1024
    scale = min(1.0, max_side / max(img.size))
    if scale < 1.0:
        img = img.resize(
            (int(img.width * scale), int(img.height * scale)), Image.BILINEAR
        )
    boxes = [
        RBox(
            ann.rbox.cx * scale,
            ann.rbox.cy * scale,
            ann.rbox.width * scale,
            ann.rbox.height * scale,
            ann.rbox.angle,
        )
        for ann in sample.annotations[:80]
    ]
    drawn = draw_boxes(img.copy(), boxes, width=2)
    ax.imshow(drawn)
    ax.set_title(f"{sample.image_path.name}\n{len(sample.annotations)} objs")
    ax.axis("off")

plt.tight_layout()
plt.show()


## 4. Optional subset (symlinks)

Builds a tiny Kaggle-shaped tree under working storage so convert/tile stay small.


In [ ]:
def build_symlink_subset(src_root: Path, dst_root: Path, n: int) -> Path:
    from oriented_det.data.fair1m import discover_fair1m_pairs

    pairs_all = sorted(discover_fair1m_pairs(src_root, "train"), key=lambda t: t[0].stem)
    pairs_all = [(img, xml) for img, xml in pairs_all if xml is not None][: int(n)]
    if len(pairs_all) < 2:
        raise ValueError("Need at least 2 labeled images for a train/val holdout")

    if dst_root.exists():
        shutil.rmtree(dst_root)
    img_out = dst_root / "Dataset" / "Images" / "Train"
    lbl_out = dst_root / "Dataset" / "Labels" / "Train"
    img_out.mkdir(parents=True)
    lbl_out.mkdir(parents=True)

    for img_path, xml_path in pairs_all:
        if xml_path is None:
            raise RuntimeError(f"Missing XML for {img_path.name}")
        os.symlink(img_path.resolve(), img_out / img_path.name)
        # Same-stem label name so the subset tree works without Notebook_Working.
        os.symlink(xml_path.resolve(), lbl_out / f"{img_path.stem}.xml")
    print(f"Subset: {len(pairs_all)} pairs → {dst_root}")
    return dst_root

if SUBSET_IMAGES is not None:
    EXPORT_ROOT = build_symlink_subset(DATA_ROOT, WORK / "fair1m_subset", SUBSET_IMAGES)
else:
    EXPORT_ROOT = DATA_ROOT
    print("Using full dump:", EXPORT_ROOT)


## 5. Convert → DOTA PNG + image-level holdout

Split **images** before tiling so overlapping tiles cannot leak across train/val.


In [ ]:
from oriented_det.data.fair1m import export_fair1m_to_dota

if DOTA_DIR.exists():
    shutil.rmtree(DOTA_DIR)

counts = export_fair1m_to_dota(
    EXPORT_ROOT,
    DOTA_DIR,
    splits=("train", "val"),
    difficult_strategy="keep",
    val_fraction=VAL_FRACTION,
    split_seed=SPLIT_SEED,
)
print("Exported:", counts)
print("ImageSets:", sorted((DOTA_DIR / "ImageSets").glob("*.txt")))


## 6. Tile (1024 / overlap 200)


In [ ]:
odet = shutil.which("odet")
if not odet:
    raise RuntimeError("odet not on PATH — reinstall oriented-det")

for split in ("train", "val"):
    split_dir = DOTA_DIR / split
    cmd = [
        odet, "tile-dota", str(split_dir),
        "--tile-size", str(TILE_SIZE),
        "--overlap", str(TILE_OVERLAP),
        "--min-overlap", str(MIN_BOX_OVERLAP),
        "--overwrite",
    ]
    print(" ".join(cmd))
    subprocess.check_call(cmd)

TRAIN_TILES = DOTA_DIR / "train" / f"tiles_{TILE_SIZE}"
VAL_TILES = DOTA_DIR / "val" / f"tiles_{TILE_SIZE}"
for label, d in (("train", TRAIN_TILES), ("val", VAL_TILES)):
    n_img = len(list((d / "images").glob("*.png"))) if (d / "images").is_dir() else 0
    n_txt = len(list((d / "labels").glob("*.txt"))) if (d / "labels").is_dir() else 0
    print(f"{label}: {n_img} png, {n_txt} txt → {d}")


## 7. Smoke train (1 epoch, capped samples)

Writes a local override JSON. `_base_` fragments resolve from the installed `oriented_det/configs` tree. This checks the convert → tile → train path; it is **not** the 12-epoch 36.7% run.


In [ ]:
cfg = {
    "_base_": [
        "../_base_/datasets/fair1m.json",
        "../_base_/models/oriented_rcnn_r50.json",
        "../_base_/schedules/1x.json",
        "../_base_/preprocessing.json",
    ],
    "enable_albumentation": False,
    "dataset": {
        "format": "dota",
        "data_root": str(DOTA_DIR),
        "train_tiles_dir": str(TRAIN_TILES),
        "val_tiles_dir": str(VAL_TILES),
        "overlap": TILE_OVERLAP,
        "difficult_strategy": "keep",
        "filter_empty_gt": True,
        "max_train_samples": MAX_TRAIN_SAMPLES,
        "max_val_samples": MAX_VAL_SAMPLES,
    },
    "data_loader": {
        "batch_size": BATCH_SIZE,
        "num_workers": NUM_WORKERS,
        "shuffle": True,
        "pin_memory": True,
    },
    "training": {
        "num_epochs": NUM_EPOCHS,
        "learning_rate": 0.005,
        "use_amp": bool(torch.cuda.is_available()),
    },
    "evaluation": {
        "compute_map_final": False,
        "compute_map_every_n_epochs": 0,
    },
    "checkpoint": {
        "discover_previous_run": False,
    },
}

CONFIG_PATH.write_text(json.dumps(cfg, indent=2))
print("Wrote", CONFIG_PATH)


In [ ]:
if not RUN_TRAIN:
    print("RUN_TRAIN=False — skip training")
else:
    cmd = [odet, "train", "--config", str(CONFIG_PATH), "--batch-size", str(BATCH_SIZE)]
    print(" ".join(cmd))
    subprocess.check_call(cmd)
    runs = WORK / "runs"
    print("Checkpoints:")
    for p in sorted(runs.rglob("*.pth"))[:20]:
        print(" ", p)


## Expected mAP (full 1× Faster R-CNN)

A complete 12-epoch finetune of [`configs/rotated_faster_rcnn/fair1m_le90_1x.json`](https://github.com/DL4EO/oriented-det/blob/main/configs/rotated_faster_rcnn/fair1m_le90_1x.json) from `hf://rotated_faster_rcnn_dota_le90_1x` (37-way classifier re-init) finished at **36.70%** mAP50 on non-empty val tiles.

Run `runs/rotated_faster_rcnn/20260910-072116` (NVIDIA L4, ~24.5 h). Train 20,055 tiles / val 9,896 after `filter_empty_gt`. Periodic mAP every 4 epochs, score ≥ 0.05, rotated IoU 0.50 (exact CPU polygon). **Not** the Gaofen hidden test and **not** FAIR1M `mAP_F`. No FAIR1M Hub zoo.

| Epoch | Train loss | Val mAP50 |
|------:|-----------:|----------:|
| 4 | 0.496 | 30.03% |
| 8 | 0.456 | 33.71% |
| 12 | 0.412 | **36.70%** |

That looks low next to DOTA (same detector ~74% official Task 1 / ~83% leaky eval-val) and is **in band for FAIR1M**:

- FAIR1M paper, Faster R-CNN R101, official OBB: **31.53%**
- Later Rotated Faster R-CNN R50 papers: **~33–35%**
- Oriented R-CNN R50 (rotated proposals): **~39–42%**

The bottleneck is **class ID, not boxes**. Epoch 12 mean best IoU vs any detection was **0.62**, same-class **0.50**, GT cover **62%** (DOTA ~88%). Train imbalance is **1038×** (Small Car 143k vs C919 138). Rare subtypes stay near 0 AP (Tractor, other-ship, Trailer, ARJ21, C919); sports fields are easy (Baseball Field 88.5%, Tennis Court 81%). Loss and mAP were still moving at epoch 12.

To raise it: resume / 3×; switch to Oriented R-CNN 1×; enable `loss.roi_grouped_ce_*` or class weights for airplane / ship / vehicle subtypes. Details: [FAIR1M user guide](https://github.com/DL4EO/oriented-det/blob/main/docs/user-guide/data.md#fair1m).

## Notes

- **Holdout protocol:** sorted stems, `md5(f"{seed}:{stem}")` vs `val_fraction` (defaults seed `0`, fraction `0.1`). Lists land in `FAIR1M-dota/ImageSets/`. Official FAIR1M-1.0 already has labeled val — omit `--val-fraction` on a full dump.
- **Zoo roles:** DOTA = Hub pretrain; HRSC2016 = published small-data Hub; FAIR1M = train/metrics support only (this notebook).
- **Full run:** set `SUBSET_IMAGES = None`, raise `NUM_EPOCHS` to 12, clear sample caps, and use a published recipe (`fair1m_le90_1x.json`) rather than this smoke JSON. Prefer a machine with large disk — Kaggle working space is usually too small for the full tile set. Expect mid-30s mAP50 for Faster R-CNN 1×, not DOTA 70%+.
- Docs: [FAIR1M user guide](https://github.com/DL4EO/oriented-det/blob/main/docs/user-guide/data.md#fair1m)
